In [ ]:
import sys
print(f"Current executable: {sys.executable}")

# This will tell you if the libraries are actually installed in that specific venv
import geemap 
print("Geemap is connected.")




Current executable: c:\developer\personal projects\floatplane_lakes\venv\Scripts\python.exe
Geemap is connected.


In [5]:
import ee
# ee.Authenticate() # This will pop open a browser window to link your Google account
ee.Initialize(project='protean-iridium-480321-v3')

print("Earth Engine initialized successfully!")



Earth Engine initialized successfully!


In [14]:
import ee
import geemap




# 1. Load the ESA WorldCover 10m dataset to get land/water masks
land_cover = ee.ImageCollection("ESA/WorldCover/v100").first().clip(kenai)

# 2. Extract the water class (Value 80 is permanent water in ESA WorldCover)
# This acts as a 'filter' that excludes the ocean
is_inland = land_cover.eq(80)

# 3. Apply the mask: Only keep water where the ESA map also says it's land/inland
refined_water = water.updateMask(is_inland)

# 4. Display
m = geemap.Map(center=[60.2, -151.0], zoom=9)
m.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2')
m.addLayer(refined_water, {'palette': ['blue']}, 'Inland Lakes Only')
m


# 1. Define the area of interest (Kenai Peninsula)
kenai = ee.Geometry.Rectangle([-152, 59.3, -148.5, 61.4])

# 2. Get Sentinel-2 imagery for the current summer season
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(kenai) \
    .filterDate('2026-06-01', '2026-07-17') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median() \
    .clip(kenai)

# 3. Calculate NDWI (Green - NIR) / (Green + NIR)
# B3 = Green, B8 = NIR
ndwi = s2.normalizedDifference(['B3', 'B8'])

# 4. Mask: Keep pixels where NDWI > 0.2 (usually indicates surface water)
water = ndwi.updateMask(ndwi.gt(0.2))

# 5. Display
m = geemap.Map(center=[60.2, -151.0], zoom=9)
m.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 RGB')
m.addLayer(water, {'palette': ['blue']}, 'Detected Water')
m

Map(center=[60.2, -151.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [37]:
import ee
import geemap

# 1. Setup Data (Same as before)
kenai = ee.Geometry.Rectangle([-151, 60, -150, 61])
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(kenai).filterDate('2026-06-01', '2026-07-17') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)).median().clip(kenai)

# 2. Refined Water Mask
ndwi = s2.normalizedDifference(['B3', 'B8'])
water_mask = ndwi.gt(0.2).selfMask() 

# 3. Create Unique IDs for each lake "blob"
# connectedComponents finds contiguous shapes
object_id = water_mask.connectedComponents(ee.Kernel.plus(1), 256)

# 4. Calculate Area (Square Meters)
# pixelArea() gives each pixel its size in m^2; reduceRegion sums them per object
area_img = ee.Image.pixelArea().addBands(object_id).reduceConnectedComponents(
    ee.Reducer.sum(), 'labels', 256
)

# 5. Define Size Constraints (in square meters)
MIN_SIZE = 50000   # ~5 hectares (Minimum safe landing length)
MAX_SIZE = 10000000 # ~10 sq km (Exclude large bodies like Kenai Lake/Ocean)

# 6. Filter by size AND remove Ocean (using WorldCover)
land_cover = ee.ImageCollection("ESA/WorldCover/v100").first().clip(kenai)
is_inland = land_cover.eq(80)

final_lakes = area_img.updateMask(
    area_img.gte(MIN_SIZE).And(area_img.lte(MAX_SIZE)).And(is_inland)
)

# 7. Display
m = geemap.Map(center=[60.2, -151.0], zoom=9)
m.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2')
m.addLayer(final_lakes.randomVisualizer(), {}, 'Qualified Floatplane Lakes')
m

Map(center=[60.2, -151.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [39]:
import ee
import geemap

# Initialize GEE
ee.Initialize()

# 1. Define Area (Kenai Peninsula subset)
kenai = ee.Geometry.Rectangle([-151, 60.5, -150.0, 61])

# 2. Get Imagery (Sentinel-2 Harmonized)
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(kenai) \
    .filterDate('2025-06-01', '2025-09-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median() \
    .clip(kenai)

# 3. Water Masking (Liberal threshold for glacial/turbid water)
ndwi = s2.normalizedDifference(['B3', 'B8'])
# Threshold set to -0.1 to catch very silty water
water = ndwi.gte(-0.1) \
            .focal_max(radius=3, kernelType='circle') \
            .focal_min(radius=1, kernelType='circle') \
            .selfMask()

# 4. Vectorize with forced projection
# Using s2.projection() ensures the vector coordinates match the pixel grid
lake_vectors = water.reduceToVectors(
    geometry=kenai,
    crs=s2.projection(), 
    scale=30, 
    geometryType='polygon',
    eightConnected=True,
    bestEffort=True,
    maxPixels=1e13
)

# 5. Clean Up (Filter out noise)
# We use maxError=1 to ensure the geometry area calculation doesn't fail
# clean_lakes = lake_vectors.map(lambda f: f.set('area', f.geometry().area(maxError=1))) \
#                          .filter(ee.Filter.gt('area', 5000))




def add_metrics(feature):
    geom = feature.geometry()
    
    # Get the bounding box coordinates [[x1, y1], [x2, y2], ...]
    # This is much safer than manual distance calculations
    bounds = geom.bounds().coordinates().get(0)
    
    # Extract min/max X and Y to find width and height of the bounding box
    # This acts as a proxy for the 'straightness' of the feature
    xs = ee.List(bounds).map(lambda pt: ee.List(pt).get(0))
    ys = ee.List(bounds).map(lambda pt: ee.List(pt).get(1))
    
    width = ee.Number(xs.reduce(ee.Reducer.max())).subtract(xs.reduce(ee.Reducer.min()))
    height = ee.Number(ys.reduce(ee.Reducer.max())).subtract(ys.reduce(ee.Reducer.min()))
    
    # Calculate approximate distance in meters (using project to Web Mercator)
    # We project to EPSG:3857 to ensure measurements are in meters
    width_m = width.multiply(111320).multiply(ee.Number(geom.centroid().get(1)).cos())
    length_m = height.multiply(111320)
    
    # Determine which is the "longest" dimension
    long_dim = length_m.max(width_m)
    short_dim = length_m.min(width_m)
    
    # Elevation
    elevation = dem.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom.centroid(),
        scale=30
    ).get('elevation')
    
    return feature.set({
        'length_m': long_dim, 
        'width_m': short_dim,
        'elevation': elevation
    })

# Relaxed Filtering: 
# Keep it if it's long enough (>800m) AND wide enough for a plane (>40m)
navigable_water = lake_vectors.map(add_metrics) \
    .filter(ee.Filter.gt('length_m', 800)) \
    .filter(ee.Filter.gt('width_m', 40)) \
    .filter(ee.Filter.lt('elevation', 760))



# --- DEBUGGING ---
# This will tell you exactly why nothing appears on the map
lake_count = clean_lakes.size().getInfo()
print(f"DEBUG: Lakes detected: {lake_count}")

# 6. Display
m = geemap.Map(center=[60.7, -150.5], zoom=11)
m.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Satellite')

# Use standard addLayer if style() was causing display issues
if lake_count > 0:
    m.addLayer(clean_lakes, {'color': 'cyan'}, 'Inland Lakes')
else:
    print("WARNING: No lakes found. Try relaxing the NDWI threshold in Step 3.")

m

AttributeError: 'Geometry' object has no attribute 'get'

In [44]:
import ee
import geemap

# Initialize Earth Engine
ee.Initialize()

# 1. Configuration & AOI (Kenai Peninsula)
kenai_aoi = ee.Geometry.Rectangle([-150.7, 60.6, -150.2, 61.0])
dem = ee.Image("USGS/SRTMGL1_003").clip(kenai_aoi)

# 2. Data Acquisition (Sentinel-2 Harmonized)
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(kenai_aoi) \
    .filterDate('2025-06-01', '2025-09-01') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median() \
    .clip(kenai_aoi)

# 3. Water Masking (NDWI)
# NDWI = (Green - NIR) / (Green + NIR)
ndwi = s2.normalizedDifference(['B3', 'B8'])
water_mask = ndwi.gt(0.2)  # Threshold for open water

# Convert mask to vector polygons
lake_vectors = water_mask.selfMask().reduceToVectors(
    geometry=kenai_aoi,
    crs=s2.projection(),
    scale=30,
    geometryType='polygon',
    eightConnected=False,
    labelProperty='zone',
    reducer=ee.Reducer.countEvery()
)

# 4. Geometry Metrics Function
def add_metrics(feature):
    """Calculates length, width, and elevation for water landing suitability."""
    geom = feature.geometry()
    bounds = geom.bounds().coordinates().get(0)
    coords_list = ee.List(bounds)
    
    xs = coords_list.map(lambda pt: ee.List(pt).get(0))
    ys = coords_list.map(lambda pt: ee.List(pt).get(1))
    
    min_x, max_x = ee.Number(xs.reduce(ee.Reducer.min())), ee.Number(xs.reduce(ee.Reducer.max()))
    min_y, max_y = ee.Number(ys.reduce(ee.Reducer.min())), ee.Number(ys.reduce(ee.Reducer.max()))
    
    # Calculate dimensions in meters
    width_m = ee.Geometry.Point([min_x, min_y]).distance(ee.Geometry.Point([max_x, min_y]))
    length_m = ee.Geometry.Point([min_x, min_y]).distance(ee.Geometry.Point([min_x, max_y]))
    
    # Elevation at centroid
    elevation = dem.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom.centroid(maxError=10),
        scale=30,
        bestEffort=True
    ).get('elevation')
    
    return feature.set({
        'length_m': length_m.max(width_m), 
        'width_m': length_m.min(width_m),
        'elevation': ee.Algorithms.If(elevation, elevation, 0)
    })

# 5. Apply Safety Filters
# >800m length, >40m width, <760m elevation (approx 2500ft)
navigable_water = lake_vectors.map(add_metrics) \
    .filter(ee.Filter.gt('length_m', 800)) \
    .filter(ee.Filter.gt('width_m', 40)) \
    .filter(ee.Filter.lt('elevation', 760))

# 6. Visualization
m = geemap.Map()
m.centerObject(kenai_aoi, 9)
m.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2')
m.addLayer(navigable_water, {'color': 'cyan'}, 'Navigable Water')
m

Map(center=[60.79981588248508, -150.45000000000027], controls=(WidgetControl(options=['position', 'transparent…